# Task 3: Category articles with Llama 3.2 3B Instruct

**Plan**
1. pandas computes the facts (top 3 products, ratings, worst product).
2. Llama reads a few reviews per product and extracts praise and complaints.
3. Llama writes the article from the fact sheet only (no invented facts).
4. We compare 3 prompt variants and save every article as markdown.

**Before you start**
- Use a GPU runtime (Colab T4 is enough for a 3B model).
- Llama is a gated model: open https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct, accept the license, and create a read token at https://huggingface.co/settings/tokens.
- Input: the CSV from Task 2 (`reviews_with_meta_category.csv`). Adjust the column names in the settings cell.

In [32]:
%pip install -q transformers accelerate torch pandas huggingface_hub

## 1. Log in to Hugging Face
Type the token when prompted. It is not saved in the notebook.

In [33]:
from getpass import getpass
from huggingface_hub import login

login(token=getpass("Hugging Face token: "))

Hugging Face token: ··········


## 2. Imports and settings

In [34]:
import os
import glob
import functools

import pandas as pd
import torch
from transformers import pipeline

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

CSV_PATH = "./data/reviews_with_meta_categories.csv"
NAME_COL = "name"               # product name
CAT_COL = "meta_category"       # category created in Task 2
TEXT_COL = "reviews.text"       # review text
RATING_COL = "reviews.rating"   # 1-5 stars

OUT_DIR = "articles_llama"
os.makedirs(OUT_DIR, exist_ok=True)

print("GPU available:", torch.cuda.is_available())

GPU available: True


## 3. Load the data


In [35]:
#!git clone https://github.com/zinaelnahel/NLP-automated-customer-reviews.git
#%cd NLP-automated-customer-reviews

df = pd.read_csv(CSV_PATH)
print(df.columns.tolist())

df = df.dropna(subset=[NAME_COL, CAT_COL, TEXT_COL, RATING_COL])
print(len(df), "reviews,", df[NAME_COL].nunique(), "products,", df[CAT_COL].nunique(), "categories")
df[CAT_COL].value_counts()

Cloning into 'NLP-automated-customer-reviews'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 101 (delta 37), reused 76 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 4.16 MiB | 2.55 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/NLP-automated-customer-reviews/NLP-automated-customer-reviews
['id', 'name', 'asins', 'brand', 'categories', 'keys', 'manufacturer', 'reviews.date', 'reviews.dateAdded', 'reviews.dateSeen', 'reviews.didPurchase', 'reviews.doRecommend', 'reviews.id', 'reviews.numHelpful', 'reviews.rating', 'reviews.sourceURLs', 'reviews.text', 'reviews.title', 'reviews.userCity', 'reviews.userProvince', 'reviews.username', 'cluster_id', 'learned_cluster_id', 'meta_category']
27867 reviews, 48 products, 5 categories


/tmp/ipykernel_2113/642781093.py:4: DtypeWarning: Columns (1,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


,count
meta_category,
Tablets and e-readers,21475
Echo speakers and smart-home devices,5965
"Chargers, cables and power accessories",401
Cases and protective covers,13
Fire TV and streaming devices,13


## 4. Load Llama 3.2 3B Instruct
float16 uses about 6-7 GB of GPU memory, which fits on a free T4 (15 GB).

In [36]:
generator = pipeline(
    "text-generation",
    model=MODEL_ID,
    torch_dtype=torch.float16,   # newer transformers versions may call this `dtype`
    device_map="auto",
)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

## 5. Helper: send a chat prompt to the model
`do_sample=False` makes the output deterministic, so differences between prompt variants come from the prompt, not from randomness.

In [37]:
def chat(system, user, max_new_tokens=500):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    out = generator(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    return out[0]["generated_text"][-1]["content"].strip()


# Quick test
print(chat("You are a helpful assistant.", "Say hello in one short sentence.", 30))

[transformers] Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hello.


## 6. Rank products inside each category
A plain average rating favors products with only a few 5-star reviews. The weighted score pulls products with few reviews toward the global average.

In [38]:
stats = (
    df.groupby([CAT_COL, NAME_COL])
    .agg(n_reviews=(RATING_COL, "size"), avg_rating=(RATING_COL, "mean"))
    .reset_index()
)

global_mean = df[RATING_COL].mean()
m = stats["n_reviews"].quantile(0.5)   # confidence threshold

stats["score"] = (
    stats["n_reviews"] / (stats["n_reviews"] + m) * stats["avg_rating"]
    + m / (stats["n_reviews"] + m) * global_mean
)

stats.sort_values([CAT_COL, "score"], ascending=[True, False]).head(10)

,meta_category,name,n_reviews,avg_rating,score
2,Cases and protective covers,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,2,4.500000,4.548835
4,Cases and protective covers,"Kindle Keyboard,,,\nKindle Keyboard,,,",2,4.500000,4.548835
1,Cases and protective covers,"Amazon Kindle Lighted Leather Cover,,,\nKindle...",1,4.000000,4.484382
0,Cases and protective covers,"Amazon Kindle Lighted Leather Cover,,,\nAmazon...",2,3.500000,4.298835
3,Cases and protective covers,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",6,3.666667,4.115890
7,"Chargers, cables and power accessories","Amazon Kindle Fire Hd (3rd Generation) 8gb,,,\...",1,4.000000,4.484382
6,"Chargers, cables and power accessories",Amazon 5W USB Official OEM Charger and Power A...,188,4.452128,4.455622
5,"Chargers, cables and power accessories","All-New Kindle E-reader - Black, 6 Glare-Free ...",212,4.429245,4.432985
11,Echo speakers and smart-home devices,"Amazon Fire Hd 10 Tablet, Wi-Fi, 16 Gb, Specia...",128,4.773438,4.764110
8,Echo speakers and smart-home devices,Amazon - Amazon Tap Portable Bluetooth and Wi-...,318,4.729560,4.726514


## 7. Extract praise and complaints per product (LLM reads a small sample)
`lru_cache` stores results so each product is only processed once, even though we test three prompt variants.

In [39]:
@functools.lru_cache(maxsize=None)
def summarize_reviews(product, kind):
    sub = df[df[NAME_COL] == product]
    if kind == "complaints":
        sub = sub[sub[RATING_COL] <= 2]
        task = "the 3 most common complaints"
    else:
        sub = sub[sub[RATING_COL] >= 4]
        task = "the 2 things reviewers praise most"

    if len(sub) == 0:
        return "Nothing notable."

    sample = sub[TEXT_COL].astype(str).str.slice(0, 300).head(15).tolist()
    reviews = "\n".join(f"- {r}" for r in sample)

    return chat(
        "You summarize customer reviews. Use only what the reviews say. Do not add anything else.",
        f"Reviews:\n{reviews}\n\nList {task} as short bullet points.",
        max_new_tokens=150,
    )

## 8. Build the fact sheet for one category
Everything in it (names, ratings, counts) comes from pandas. The LLM only contributes the praise and complaint bullets.

In [40]:
def build_facts(category, min_reviews=5):
    cat = stats[(stats[CAT_COL] == category) & (stats["n_reviews"] >= min_reviews)]
    top = cat.nlargest(3, "score")
    worst = cat.nsmallest(1, "score")

    lines = [f"Category: {category}", "", "Top products:"]
    for i, row in enumerate(top.itertuples(), 1):
        name = row[2]   # columns are: Index, category, product name, n_reviews, ...
        lines.append(f"{i}. {name[:80]} | average rating {row.avg_rating:.1f} from {row.n_reviews} reviews")
        lines.append(f"   Praised for: {summarize_reviews(name, 'praise')}")
        lines.append(f"   Complaints: {summarize_reviews(name, 'complaints')}")

    for row in worst.itertuples():
        name = row[2]   # columns are: Index, category, product name, n_reviews, ...
        lines.append("")
        lines.append(f"Worst product: {name[:80]} | average rating {row.avg_rating:.1f} from {row.n_reviews} reviews")
        lines.append(f"   Complaints: {summarize_reviews(name, 'complaints')}")

    return "\n".join(lines)

## 9. Prompt variants to compare
This is the prompt-engineering experiment the brief asks for.

In [41]:
SYSTEM = "You are a writer for a consumer-electronics review website."

PROMPT = (
    "Write a blog post of 250-350 words using ONLY the facts below.\n"
    "Structure: a title, a 2-sentence intro, a section 'Top 3 picks' with the key "
    "differences between the products, a section 'Common complaints', and a section "
    "'Which one to avoid'. Do not invent products, numbers or features.\n\n{facts}"
)

## 10. Test on ONE category first
Check the fact sheet and the three articles before running everything.

In [42]:
test_category = sorted(df[CAT_COL].unique())[0]
facts = build_facts(test_category)
print(facts)
print("\n" + "=" * 60 + "\n")

article = chat(SYSTEM, PROMPT.format(facts=facts), max_new_tokens=600)
print(article)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Category: Cases and protective covers

Top products:
1. Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid-Proof Case | average rating 3.7 from 6 reviews
   Praised for: • Responsiveness and performance
• Affordability and value for money
   Complaints: • Limited app selection and slow browsing
• Poor performance, freezing, and restarting issues
• Charging issues with the provided charging cable connector

Worst product: Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid-Proof Case | average rating 3.7 from 6 reviews
   Complaints: • Limited app selection and slow browsing
• Poor performance, freezing, and restarting issues
• Charging issues with the provided charging cable connector


**Protecting Your Kids' Devices: A Review of Cases and Protective Covers**

As a parent, it's essential to ensure your child's device is protected from accidental damage and harsh environments. In this review, we'll be focusing on cases and protective covers for kids' tablets, highlig

## 11. Generate articles for all categories and variants
Each article is saved as a markdown file in `articles_llama/`.

In [43]:
# Ensure the output directory exists in our current working directory
os.makedirs(OUT_DIR, exist_ok=True)

for category in sorted(df[CAT_COL].unique()):
    facts = build_facts(category)
    safe = str(category).lower().replace(" ", "_").replace("/", "_")

    article = chat(SYSTEM, PROMPT.format(facts=facts), max_new_tokens=600)
    with open(f"{OUT_DIR}/{safe}.md", "w", encoding="utf-8") as f:
        f.write(article)
    print(f"saved: {category}")

[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


saved: Cases and protective covers


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

saved: Chargers, cables and power accessories


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

saved: Echo speakers and smart-home devices


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


saved: Fire TV and streaming devices


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

saved: Tablets and e-readers
